# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 3: Decoder-only Transformers</font>

# <font color="#003660">Notebook 2: Domain Adaptation of a Causal Language Model</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>
<center>
<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... will be able to fine-tune a causal language model on your own data, which is useful for training decoder models.
    </font>
</div>
</center>
</p>

The following content is heavily inspired by the following excellent sources:


*   Tunstall et al. (2022): Natural Language Processing with Transformers. O'Reilly. https://www.oreilly.com/library/view/natural-language-processing/9781098103231/
*   Hugging Face: LLM Course. https://huggingface.co/learn/llm-course



# Recall: Transfer Learning and Causal Language Modeling

There are different strategies for transfer learning with language models.
- The first approach is **pre-training** a language model on a large corpus of unlabeled text data using masked or causal language modeling objectives.
- The second approach is **domain-adaptive pre-training (DAPT)**, where a language model is further pretrained on unlabeled data from the target domain.
- The third approach is **fine-tuning** the pretrained/domain-adapted language model on a smaller labeled dataset for a specific downstream task. This could be a supervised task, such as text classification or named entity recognition. Or it could be an unsupervised task, such as text generation in a specific style or format (e.g., question answering or dialogue).

The figure below illustrates this process, which was first proposed by [Howard and Ruder in 2018](https://arxiv.org/abs/1801.06146).

<center><img width=600 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/ulmfit.png"/><br></center>

Language models are pre-trained (and domain-adapted) with one of two training objectives:

- **Masked language modeling (MLM):** the model predicts missing (masked) tokens anywhere in a sequence, using the tokens to their left *and* right. This objective is used to train **encoder models** like BERT (Session 2).

<center><img width=300 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/mlm.png"/></center>

- **Causal language modeling (CLM):** the model predicts the next token, using only the tokens to its left. This objective is used to train **decoder models** like GPT.

<center><img width=400 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/clm.png"/></center>

In this notebook, we domain-adapt a causal language model. Domain adaptation of an encoder model with MLM works exactly the same way, with only one difference: instead of predicting the next token, a *data collator* randomly replaces 15% of the tokens with a `[MASK]` token, and the model learns to restore them.

# Import Packages

In [ ]:
import pandas as pd
import numpy as np
import math
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from transformers import TrainingArguments
from transformers import Trainer

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Load Pre-trained Model

First, we load a model for causal language modeling and a corresponding tokenizer from the model hub.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
device

In [ ]:
model_name = "distilbert/distilgpt2"

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Testdrive the Model 🚗

In [ ]:
input_txt = "Bob and Clara are great"

In [ ]:
inputs = tokenizer(input_txt, return_tensors="pt").to(device)
torch.manual_seed(42)  # same seed before and after domain adaptation, for a fair comparison
output = model.generate(**inputs, max_new_tokens=40, do_sample=True, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(output[0]))

# Prepare a Dataset for Domain Adaptation

Load the IMDB movie reviews. We only need the 25,000 reviews of the training split: tokenizing the test split and the 50,000 unlabeled reviews as well would only cost time.

In [ ]:
imdb_dataset = load_dataset("stanfordnlp/imdb", split="train")
imdb_dataset

In [ ]:
def tokenize_function(examples):
    result = tokenizer(examples["text"])
    return result

In [ ]:
tokenized_datasets = imdb_dataset.map(
    tokenize_function, batched=True, remove_columns=["text", "label"]
)
tokenized_datasets

For language modeling, a [common preprocessing step](https://youtu.be/8PmhEIXhBvI) is to concatenate all the samples and then split the resulting text into chunks of context length. This way, we can get around the usual  padding and truncating of individual samples and make sure that the model sees the same amount of context for each sample.

The function below, taken from the Hugging Face LLM Course (https://huggingface.co/learn/llm-course/chapter7/3), does exactly this, and some other preprocessing steps.

In [ ]:
chunk_size = 128

def group_texts(examples):
    # Concatenate all texts
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    # Compute length of concatenated texts
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the last chunk if it's smaller than chunk_size
    total_length = (total_length // chunk_size) * chunk_size
    # Split by chunks of max_len
    result = {
        k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
        for k, t in concatenated_examples.items()
    }
    # Create a new labels column
    result["labels"] = result["input_ids"].copy()
    return result

In [ ]:
lm_datasets = tokenized_datasets.map(group_texts, batched=True)
lm_datasets

In [ ]:
lm_datasets[1]["input_ids"][0:10]

In [ ]:
lm_datasets[1]["labels"][0:10]

# Domain-Adapt with Trainer API

Let's draw a sample of the dataset, so that we don't have to wait too long.

In [ ]:
train_size = 5000
test_size = int(0.1 * train_size)

downsampled_dataset = lm_datasets.train_test_split(
    train_size=train_size, test_size=test_size, seed=42
)

downsampled_dataset

For causal language modeling, we don't need a data collator. During training, the labels will be automatically shifted by one position so that the task is to predict the token at timestep `t+1`, using all tokens up to `t`.

In [ ]:
batch_size = 32  # larger batches need a lot of (GPU) memory: the model outputs a score for each of the 50,257 vocabulary tokens at each position
logging_steps = len(downsampled_dataset["train"]) // batch_size

training_args = TrainingArguments(
    output_dir="distilgpt2-clm-finetuned-imdb",
    num_train_epochs=1,
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    logging_steps=logging_steps,
    fp16=torch.cuda.is_available(),  # mixed precision speeds up training, but only works on (NVIDIA) GPUs
    save_strategy="no",
    report_to="none",
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=downsampled_dataset["train"],
    eval_dataset=downsampled_dataset["test"]
)

Perplexity of the pre-trained, but not domain-adapted model.

In [ ]:
eval_results = trainer.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Perform domain adaptation! On a Colab GPU, this takes a few minutes. Make sure that you have selected a GPU runtime (*Runtime > Change runtime type > T4 GPU*); on a CPU it takes much longer.

If you don't want to wait, set `train_model = False`. Then we skip the training and load a model that has already been domain-adapted with exactly the same code from the Hugging Face Hub.

In [ ]:
train_model = True
adapted_model_name = "olivermueller/distilgpt2-clm-finetuned-imdb"

In [ ]:
if train_model:
    trainer.train()
else:
    model = AutoModelForCausalLM.from_pretrained(adapted_model_name).to(device)
    trainer = Trainer(model=model, args=training_args, eval_dataset=downsampled_dataset["test"])

Calculate the perplexity of the domain-adapted model. Perplexity measures how “surprised” the model is by the evaluation texts (lower is better): it is the exponential of the average loss per token.

In [ ]:
eval_results = trainer.evaluate()
print(f">>> Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

# Testdrive the Domain-adapted Model 🛫

Let's see if the text generated by the domain-adapted model differs from the text generated by the original model.

In [ ]:
model.eval()  # switch off dropout, which the Trainer has switched on for training
input_txt = "Bob and Clara are great"
inputs = tokenizer(input_txt, return_tensors="pt").to(device)
torch.manual_seed(42)
output = model.generate(**inputs, max_new_tokens=40, do_sample=True, pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(output[0]))

# Exercises

1. **Another prompt:** Test the original and the domain-adapted model with a prompt that fits the movie domain (e.g., "The acting in this film") and one that doesn't (e.g., "The stock market"). How does domain adaptation change the generated texts?
2. **More training:** Increase `train_size` or the number of epochs (`num_train_epochs` in `TrainingArguments`). How far does the perplexity go down?
3. **Your own domain:** Domain-adapt the model to the wine reviews from Session 1: load `wine_reviews_train.csv` with pandas, convert the `description` column into a dataset with `Dataset.from_pandas()`, and run the same pipeline. Can you make the model write like a sommelier?

In [ ]:
# YOUR CODE GOES HERE